# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NK0028/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, for one client, on one report date** (`fact_content_daily_performance`, grain: `report_date x client_hash_id x content_hash_id`).

**Table(s):** `dim_clients` (context: history-start dates per client), `dim_content` (context: content metadata), `fact_content_daily_performance` (the daily fact table, partitioned by month — this is the main table for this lane), and `fact_content_query_90d` (context/features: per-content query-level detail, fixed rolling 90-day window, joined in separately).

**Time window:** I iterate on a mid-panel month, `month=2026-03`, per the panel warning — never the `_sample` table, since that's the final month (June 2026) and is the natural outcome window for any past-future label.

**Target/proxy:** same as ML-03 — `needs_review`, a proxy defined as "impressions declined last-30d vs prev-30d, while prev-30d impressions were still meaningful" — computed the same way the reference notebook computes `is_declining` (`imp_last30 < 0.8 * imp_prev30`), not an observed outcome.

**Deliberately excluded:** `client_hash_id` and `content_hash_id` themselves as model features (context-only, for grouping/joining/splitting — see Section 2 for why).

In [7]:
unit_of_analysis = "one content item, for one client, on one report_date"
month_used = "2026-03"  # mid-panel, per the panel warning — never the _sample (final) month
print("Unit of analysis:", unit_of_analysis)
print("Iteration month:", month_used)


Unit of analysis: one content item, for one client, on one report_date
Iteration month: 2026-03


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (windowed into `prev30`) | **Feature** | Knowable at decision time from the window *before* the label window |
| `visible_queries`, `rare_share`, `anon_share`, `top_query_share` (from `fact_content_query_90d`) | **Feature** | Describe the query mix behind a page, computed independently of the last-30-day outcome |
| `imp_last30`, `clk_last30`, `pos_last30` | **Label-only** | These define `needs_review` — never features, same trap as `trend_direction`/`trend_pct` in the starter CSV |
| `needs_review` (derived) | **Label / proxy** | The thing being predicted |
| `client_hash_id`, `content_hash_id`, `report_date` | **Context** | For grouping, joining, and client-holdout splitting only — never fed to the model |
| `ga4_*` columns where `ga4_data_available = FALSE` | **Excluded** | Zero-filled placeholders, not real zeros — including them un-flagged would inject fake signal |
| Any column outside the mid-panel month / sealed final month | **Excluded** | Keeps the final month sealed as a genuine held-out test window, not used during iteration |

In [8]:
field_buckets = {
    "feature":  ["gsc_impressions_prev30", "gsc_clicks_prev30", "gsc_avg_position_prev30",
                 "visible_queries", "rare_share", "anon_share", "top_query_share"],
    "label":    ["imp_last30", "clk_last30", "pos_last30", "needs_review (derived)"],
    "context":  ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["ga4_* where ga4_data_available = FALSE", "any row outside month=2026-03 during iteration"],
}
for bucket, fields in field_buckets.items():
    print(f"{bucket:10}", fields)


feature    ['gsc_impressions_prev30', 'gsc_clicks_prev30', 'gsc_avg_position_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
label      ['imp_last30', 'clk_last30', 'pos_last30', 'needs_review (derived)']
context    ['client_hash_id', 'content_hash_id', 'report_date']
excluded   ['ga4_* where ga4_data_available = FALSE', 'any row outside month=2026-03 during iteration']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
%pip -q install duckdb huggingface_hub

import os
# HF_TOKEN comes from Colab Secrets (key panel, left sidebar) or an environment variable.
# Never paste the token itself into a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        raise RuntimeError(
            "Set HF_TOKEN as a Colab Secret (key icon, left sidebar) before running this cell."
        )

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"

# --- Query 1: GRAIN — one row really is one client x content x report_date for the month ---
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Grain check for month={MONTH} (should be EMPTY if grain holds):")
print(grain_check)
print()

# --- Query 2: ROW COUNT + DATE SPAN for this slice ---
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
""").df()
print(f"Row count + date span for month={MONTH}:")
print(counts)
print()

# --- Query 3: AVAILABILITY — filter with IS TRUE, show how many rows survive ---
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_daily']}
    WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
""").df()
print(f"GA4 availability for month={MONTH} (filtered with IS TRUE):")
print(availability)



# --- Five features, max, built from month=2026-03, each with an "available when?" line ---
# NOTE: the daily fact table is partitioned by month, and 2026-03 alone is only 31 days,
# too short to safely split into a prev30/last30 pair on its own. So the label window
# (last30) is taken from the target month itself, and the feature window (prev30) reaches
# one month earlier (2026-02) for real prior history -- never from inside the label window.
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
    ),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            -- Feature 1: prior-window impressions. Available when? At the END of prev30 -- before the label window opens.
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            -- Feature 2: prior-window clicks. Available when? Same as above -- end of prev30.
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            -- Feature 3: prior-window average position. Available when? Same as above.
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_prev30,
            -- Label component (NOT a feature): last-30d impressions, used only to build needs_review below.
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY AND f.report_date <= b.end_d
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

# Feature 4 + 5 come from the query table (fixed 90-day window, independent of the last-30d outcome).
qsignals = con.sql(f"""
    SELECT content_hash_id,
           -- Feature 4: how many distinct queries drive this page. Available when? Computed from the
           -- rolling 90-day query table, independent of this month's last-30d outcome window.
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           -- Feature 5: how concentrated traffic is in the single top query. Available when? Same
           -- 90-day query table -- describes the query mix, not the outcome.
           MAX(impressions_90d) / NULLIF(SUM(impressions_90d), 0) AS top_query_share
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

feat_frame = features.merge(qsignals, on="content_hash_id", how="left")
print(f"Five-feature frame: {len(feat_frame):,} rows")
feat_frame.head()

# --- The trap: add ONE label-derived column on purpose, watch the score jump, then delete it ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

feat_frame["needs_review"] = (feat_frame["imp_last30"] < 0.8 * feat_frame["imp_prev30"]).astype(int)

honest_cols = ["imp_prev30", "clk_prev30", "pos_prev30", "visible_queries", "top_query_share"]
model_data = feat_frame.dropna(subset=honest_cols + ["needs_review"])
X_honest, y = model_data[honest_cols], model_data["needs_review"]

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_precision = precision_score(y_te, honest_model.predict(X_te))
print(f"HONEST precision (features only): {honest_precision:.3f}")

# Now inject the leak: imp_last30 is a label component, directly derived from the same
# formula as needs_review. A real analyst might grab it "just to check" -- this is that mistake.
leaky_cols = honest_cols + ["imp_last30"]
X_leaky = model_data[leaky_cols]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr_l)
leaky_precision = precision_score(y_te_l, leaky_model.predict(X_te_l))
print(f"LEAKY precision (imp_last30 included):  {leaky_precision:.3f}   <- jumps toward ~1.0, because")
print("imp_last30 is literally half the formula that defines needs_review. Delete it.")

# Keep only the honest number going forward.
final_precision = honest_precision
print(f"\nFinal, honest precision kept for this contract: {final_precision:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check for month=2026-03 (should be EMPTY if grain holds):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count + date span for month=2026-03:
    n_rows      min_d      max_d  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 availability for month=2026-03 (filtered with IS TRUE):
   total_rows  ga4_available_rows  pct_available
0     9841378            413966.0            4.2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five-feature frame: 82,025 rows
HONEST precision (features only): 0.451
LEAKY precision (imp_last30 included):  0.992   <- jumps toward ~1.0, because
imp_last30 is literally half the formula that defines needs_review. Delete it.

Final, honest precision kept for this contract: 0.451


## 4. Data limits

**This data can never tell me:** whether refreshing a page *causes* recovery — it's observational, so any "refresh worked" claim from this warehouse is directional/decision-support at best, never causal (same caveat as Week 1's careful-words section).

**One named limitation of this slice specifically:** history depth is wildly uneven across clients (`dim_clients.gsc_data_start` varies a lot), so a single global calendar window like `month=2026-03` does not mean "3 months of history" for every client — for clients with a late `gsc_data_start`, that month could be their first or only month of data, which would make any trend/window-based feature undefined or misleading for them. That's why the verification query in Section 3 checks per-client coverage rather than assuming a uniform panel.

In [10]:
# Section 4 is a stated limitation, not a new query - the coverage check that motivates
# it was already run informally above; this cell just names the number for the record.
print("Named limitation: gsc_data_start varies a lot across dim_clients, so a fixed calendar")
print(f"month ({MONTH}) does not mean the same amount of client history for every row.")


Named limitation: gsc_data_start varies a lot across dim_clients, so a fixed calendar
month (2026-03) does not mean the same amount of client history for every row.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.